# Datasheet Feature Extraction Pipeline
**Pipeline stages:**
1. MongoDB cache check (skip extraction if already stored)
2. Extract first 25 pages (text + tables)
3. Detect component type → fixed spec extraction via LLM
4. VLM graph extraction for still-missing specs
5. Expand to next 25 pages if specs still missing
6. Dynamic spec extraction as final fallback
7. Store result in MongoDB + save JSON file

In [ ]:
# ── CELL 1: Install dependencies ──────────────────────────────────────────────
!pip install pdfplumber pymupdf groq pymongo dnspython -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 92.4 MB/s eta 0:00:00


In [ ]:
# ── CELL 2: Imports ────────────────────────────────────────────────────────────
import json
import re
import os
import hashlib
import base64
import tempfile
from datetime import datetime, timezone

import pdfplumber
import fitz          # pymupdf — for rendering pages to images
from groq import Groq
from pymongo import MongoClient
from google.colab import files

print('✅ Imports OK')

✅ Imports OK


In [ ]:
# ── CELL 3: API keys & config ──────────────────────────────────────────────────
# Groq
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
# MongoDB Atlas connection string
# Format: mongodb+srv://<user>:<password>@<cluster>.mongodb.net/
# See setup instructions in the cell below
MONGO_URI = os.getenv("MONGO_URI")
MONGO_DB       = "datasheets"
MONGO_COL      = "components"

# Pipeline config
CHUNK_SIZE     = 25    # pages per extraction pass
TEXT_MODEL     = "llama-3.3-70b-versatile"
VISION_MODEL   = "meta-llama/llama-4-scout-17b-16e-instruct"   # Groq vision model

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is not set")
if not MONGO_URI:
    raise ValueError("MONGO_URI is not set")

client_groq    = Groq(api_key=GROQ_API_KEY)
client_mongo   = MongoClient(MONGO_URI)
col            = client_mongo[MONGO_DB][MONGO_COL]

print('✅ Clients ready')

✅ Clients ready


## MongoDB Atlas Setup Instructions

1. Go to https://cloud.mongodb.com and sign up free
2. Create a **free M0 cluster** (any region)
3. Under **Database Access** → Add a user with password
4. Under **Network Access** → Add IP `0.0.0.0/0` (allow all, fine for dev)
5. Click **Connect** → **Drivers** → copy the connection string
6. Replace `<password>` in the string and paste into `MONGO_URI` above

The DB and collection are created automatically on first insert.

In [ ]:
# ── CELL 4: SPEC_POOL (fixed params per component type) ────────────────────────
SPEC_POOL = {
    "Resistor":           ["Resistance", "Power Rating", "Tolerance", "Maximum Voltage", "Temperature Coefficient"],
    "Capacitor":          ["Capacitance", "Voltage Rating", "Tolerance", "ESR", "Temperature Range"],
    "Inductor":           ["Inductance", "Current Rating", "DC Resistance (DCR)", "Saturation Current", "Tolerance"],
    "Diode":              ["Forward Voltage (Vf)", "Reverse Voltage (Vr)", "Forward Current (If)", "Reverse Recovery Time", "Power Dissipation"],
    "Zener Diode":        ["Zener Voltage", "Power Rating", "Tolerance", "Dynamic Resistance", "Reverse Current"],
    "Schottky Diode":     ["Forward Voltage (Vf)", "Reverse Voltage (Vr)", "Forward Current (If)", "Reverse Leakage Current", "Switching Speed"],
    "LED":                ["Forward Voltage (Vf)", "Forward Current (If)", "Luminous Intensity", "Wavelength", "Power Dissipation"],
    "BJT":                ["Collector-Emitter Voltage (Vce)", "Collector Current (Ic)", "Gain (hFE)", "Power Dissipation", "Transition Frequency (fT)"],
    "MOSFET":             ["Drain-Source Voltage (Vds)", "Continuous Drain Current (Id)", "Rds(on)", "Gate Threshold Voltage (Vgs_th)", "Power Dissipation"],
    "IGBT":               ["Collector-Emitter Voltage (Vce)", "Collector Current (Ic)", "Gate Threshold Voltage", "Switching Frequency", "Power Dissipation"],
    "LDO Regulator":      ["Input Voltage (Vin)", "Output Voltage (Vout)", "Output Current (Iout)", "Dropout Voltage", "Quiescent Current (Iq)"],
    "DC-DC Converter":    ["Input Voltage Range", "Output Voltage", "Output Current", "Efficiency", "Switching Frequency"],
    "Buck Converter":     ["Input Voltage Range", "Output Voltage", "Output Current", "Efficiency", "Switching Frequency"],
    "Boost Converter":    ["Input Voltage Range", "Output Voltage", "Output Current", "Efficiency", "Switching Frequency"],
    "Op-Amp":             ["Gain Bandwidth Product", "Input Offset Voltage", "Slew Rate", "Supply Voltage Range", "Input Bias Current"],
    "Comparator":         ["Response Time", "Input Offset Voltage", "Supply Voltage", "Input Bias Current", "Output Type"],
    "Voltage Reference":  ["Output Voltage", "Accuracy", "Temperature Coefficient", "Output Current", "Noise"],
    "ADC":                ["Resolution", "Sampling Rate", "Input Voltage Range", "Power Consumption", "SNR"],
    "DAC":                ["Resolution", "Sampling Rate", "Output Voltage Range", "Power Consumption", "THD"],
    "Microcontroller":    ["Flash Memory", "RAM", "Clock Speed", "Operating Voltage", "Number of GPIO"],
    "Memory IC":          ["Memory Size", "Interface Type", "Access Time", "Operating Voltage", "Clock Frequency"],
    "RF Transceiver":     ["Frequency Range", "Data Rate", "Output Power", "Sensitivity", "Supply Voltage"],
    "Bluetooth Module":   ["Bluetooth Version", "Range", "Data Rate", "Supply Voltage", "Power Consumption"],
    "WiFi Module":        ["WiFi Standard", "Frequency Band", "Data Rate", "Supply Voltage", "Power Consumption"],
    "Temperature Sensor": ["Temperature Range", "Accuracy", "Supply Voltage", "Output Type", "Response Time"],
    "Pressure Sensor":    ["Pressure Range", "Accuracy", "Supply Voltage", "Output Type", "Response Time"],
    "Accelerometer":      ["Measurement Range", "Sensitivity", "Bandwidth", "Supply Voltage", "Power Consumption"],
    "Gyroscope":          ["Angular Rate Range", "Sensitivity", "Bandwidth", "Supply Voltage", "Power Consumption"],
    "Crystal Oscillator": ["Frequency", "Load Capacitance", "Frequency Stability", "Operating Temperature", "ESR"],
    "Clock Generator":    ["Output Frequency", "Jitter", "Supply Voltage", "Output Type", "Frequency Stability"],
}

In [ ]:
# ── CELL 5: PDF utilities ──────────────────────────────────────────────────────

def pdf_hash(filepath):
    """SHA-256 of the PDF bytes — used as cache key."""
    h = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()


def extract_pdf_chunk(filepath, start_page=0, end_page=25):
    """
    Extract text + tables from a specific page range [start_page, end_page).
    Returns (raw_text, tables, total_pages).
    """
    tables    = []
    raw_text  = ""

    with pdfplumber.open(filepath) as pdf:
        total_pages = len(pdf.pages)
        page_slice  = pdf.pages[start_page:min(end_page, total_pages)]

        for i, page in enumerate(page_slice):
            page_num = start_page + i + 1
            text = page.extract_text()
            if text:
                raw_text += f"\n--- Page {page_num} ---\n{text}"

            for table in (page.extract_tables() or []):
                if table:
                    tables.append({"page": page_num, "rows": table})

    return raw_text, tables, total_pages


def tables_to_text(tables):
    text = ""
    for t in tables:
        text += f"\n[Page {t['page']}]\n"
        for row in t['rows']:
            clean = [str(c).strip() if c else '' for c in row]
            text += ' | '.join(clean) + '\n'
    return text


def get_figure_pages(filepath, start_page=0, end_page=25):
    """
    Return page numbers (1-indexed) that contain embedded images/figures.
    Uses pymupdf which is faster for image detection.
    """
    figure_pages = []
    doc = fitz.open(filepath)
    for i in range(start_page, min(end_page, len(doc))):
        if doc[i].get_images(full=True):   # page has at least one image
            figure_pages.append(i + 1)     # convert to 1-indexed
    doc.close()
    return figure_pages


def render_page_to_base64(filepath, page_num_1indexed, dpi=150):
    """
    Render a PDF page to a base64 PNG string for VLM input.
    dpi=150 is good balance of quality vs token size.
    """
    doc  = fitz.open(filepath)
    page = doc[page_num_1indexed - 1]
    mat  = fitz.Matrix(dpi / 72, dpi / 72)
    pix  = page.get_pixmap(matrix=mat)
    img_bytes = pix.tobytes('png')
    doc.close()
    return base64.standard_b64encode(img_bytes).decode('utf-8')


print('✅ PDF utilities ready')

✅ PDF utilities ready


In [ ]:
# ── CELL 6: LLM helpers ────────────────────────────────────────────────────────

def safe_parse_json(raw):
    """Strip markdown fences + extract first JSON object robustly."""
    raw = raw.replace('```json', '').replace('```', '').strip()
    match = re.search(r'\{.*\}', raw, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None


def detect_component_type(table_text, raw_text):
    prompt = f"""
Identify the electronic component type from this datasheet.
Return ONLY JSON, no explanation:
{{"component_type": "..."}}

Valid examples: "Resistor", "MOSFET", "LDO Regulator", "Temperature Sensor"

Tables:
{table_text[:2000]}

Text:
{raw_text[:1500]}
"""
    resp = client_groq.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    raw  = resp.choices[0].message.content
    data = safe_parse_json(raw)
    return data.get('component_type', 'Unknown').strip() if data else 'Unknown'


def extract_fixed_specs(table_text, raw_text, component_type, spec_list):
    """
    Extract fixed spec list from text. Returns dict:
    { spec_name: {value, source} }  source = 'text_llm'
    """
    fields = '\n'.join([f'- {s}' for s in spec_list])
    prompt = f"""
Component: {component_type}

Extract these specs. Match by meaning, not exact wording.
For missing specs use null.
Also extract part_name and manufacturer.

Return ONLY JSON:
{{
  "part_name": "...",
  "manufacturer": "...",
  "specs": {{
    "Spec Name": "value with unit or null"
  }}
}}

Specs to extract:
{fields}

Tables:
{table_text[:10000]}

Text:
{raw_text[:2000]}
"""
    resp = client_groq.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1
    )
    data = safe_parse_json(resp.choices[0].message.content)
    if not data:
        return {}, None, None

    # Annotate each value with source
    # In extract_fixed_specs, replace the annotation loop:
    specs = {}
    for name, val in data.get('specs', {}).items():
        is_missing = (val is None or str(val).strip().lower() == 'null' or str(val).strip() == '')
        specs[name] = {
            'value': None if is_missing else val,
            'source': None if is_missing else 'text_llm'
        }
    return specs, data.get('part_name'), data.get('manufacturer')


def find_graph_pages(raw_text, missing_specs):
    spec_list = '\n'.join([f'- {s}' for s in missing_specs])
    prompt = f"""
You are reading a datasheet. The text below has page markers like "--- Page 5 ---".

Find the ACTUAL PDF PAGE NUMBERS (from the --- Page X --- markers) that contain
graphs/figures for these parameters:
{spec_list}

CRITICAL RULES:
- Figure numbers (Figure 7, Fig. 8) are NOT the same as PDF page numbers
- Look at the --- Page X --- markers to find where "TYPICAL CHARACTERISTICS" graphs appear
- Dropout voltage graphs will be on pages labeled "TYPICAL CHARACTERISTICS"
- Ignore cover page thumbnails
- For typical_x: use standard operating condition (e.g. IO=100mA for dropout voltage)

Return ONLY JSON:
{{"graph_pages": [
  {{"spec": "Spec Name", "pages": [actual_pdf_page_numbers], "x_axis": "x axis label", "typical_x": "value"}}
]}}

Datasheet text:
{raw_text[:15000]}
"""
    resp = client_groq.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0
    )
    data = safe_parse_json(resp.choices[0].message.content)
    return data.get('graph_pages', []) if data else []


def extract_spec_from_graph_vlm(page_b64, spec_name, x_axis, typical_x, component_type):
    """
    Targeted VLM prompt — tells the model exactly what to look for and at what x value.
    """
    prompt = f"""
This is a page from a {component_type} datasheet.

Find a graph showing: {spec_name}
The x-axis is likely: {x_axis}
Read the y-axis value ({spec_name}) at x = {typical_x}.

If multiple curves exist (e.g. different temperatures or currents), read the value
at the worst case or typical condition and note which curve.

Rules:
- Only extract if the graph is clearly visible
- Return the numeric value with units
- Use null if graph not found or unreadable

Return ONLY JSON:
{{"value": "numeric value with unit or null", "condition": "which curve/condition was read"}}
"""
    resp = client_groq.chat.completions.create(
        model=VISION_MODEL,
        messages=[{
            'role': 'user',
            'content': [
                {'type': 'image_url',
                 'image_url': {'url': f'data:image/png;base64,{page_b64}'}},
                {'type': 'text', 'text': prompt}
            ]
        }],
        temperature=0
    )
    data = safe_parse_json(resp.choices[0].message.content)
    return data if data else {'value': None, 'condition': None}


def extract_dynamic_specs(table_text, raw_text, component_type, existing_spec_names):
    """
    Dynamically find additional specs not in the fixed list.
    Returns list of {name, value, source='dynamic_llm'}
    """
    existing = ', '.join(existing_spec_names)
    prompt = f"""
Component: {component_type}

Already extracted: {existing}

Find up to 5 additional important specs with numeric values that are NOT in the above list.
These should be specs that differentiate this specific component.

Return ONLY JSON:
{{"additional_specs": [
  {{"name": "Spec Name", "value": "value with unit"}}
]}}

If nothing useful found: {{"additional_specs": []}}

Tables:
{table_text[:8000]}

Text:
{raw_text[:2000]}
"""
    resp = client_groq.chat.completions.create(
        model=TEXT_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.1
    )
    data = safe_parse_json(resp.choices[0].message.content)
    if not data:
        return []
    return [
        {'name': s['name'], 'value': s['value'], 'source': 'dynamic_llm'}
        for s in data.get('additional_specs', [])
        if s.get('name') and s.get('value')
    ]


print('✅ LLM helpers ready')

✅ LLM helpers ready


In [ ]:
# ── CELL 7: MongoDB helpers ────────────────────────────────────────────────────

def check_cache(pdf_sha256):
    """Return stored document if this PDF was already processed, else None."""
    return col.find_one({'pdf_hash': pdf_sha256}, {'_id': 0})


def save_to_db(record):
    """Upsert by pdf_hash so re-runs don't duplicate."""
    col.update_one(
        {'pdf_hash': record['pdf_hash']},
        {'$set': record},
        upsert=True
    )
    print(f"✅ Saved to MongoDB: {record['part_name']} ({record['component_type']})")


print('✅ MongoDB helpers ready')

✅ MongoDB helpers ready


In [ ]:
# ── CELL 8: Main extraction pipeline ──────────────────────────────────────────

# Replace get_missing:
def get_missing(specs):
    return [
        name for name, meta in specs.items()
        if not meta.get('value') or str(meta.get('value')).strip().lower() == 'null'
    ]


def run_pipeline(filepath):
    print(f'\n{'='*60}')
    print(f'Processing: {os.path.basename(filepath)}')
    print('='*60)

    # ── Stage 0: Cache check ───────────────────────────────────────
    sha = pdf_hash(filepath)
    print(f'\n[Stage 0] Checking cache (hash: {sha[:12]}...)')
    cached = check_cache(sha)
    if cached:
        print('  ✅ Cache HIT — skipping extraction')
        return cached
    print('  Cache miss — running full pipeline')

    # ── Stage 1: First 25 pages ────────────────────────────────────
    print(f'\n[Stage 1] Extracting pages 1–{CHUNK_SIZE}')
    raw_text, tables, total_pages = extract_pdf_chunk(filepath, 0, CHUNK_SIZE)
    table_text = tables_to_text(tables)
    print(f'  Pages: {min(CHUNK_SIZE, total_pages)}/{total_pages} | Tables: {len(tables)}')

    # ── Stage 2: Detect component type ────────────────────────────
    print('\n[Stage 2] Detecting component type')
    component_type = detect_component_type(table_text, raw_text)
    print(f'  → {component_type}')

    fixed_specs = SPEC_POOL.get(component_type, [])
    if not fixed_specs:
        print(f'  ⚠ Unknown component type — will use dynamic extraction only')

    # ── Stage 3: Fixed spec extraction from text ───────────────────
    print('\n[Stage 3] Fixed spec extraction (text + tables)')
    specs, part_name, manufacturer = extract_fixed_specs(
        table_text, raw_text, component_type, fixed_specs
    )
    missing = get_missing(specs)
    print(f'  Extracted: {len(specs) - len(missing)}/{len(fixed_specs)} | Missing: {missing}')

    # ── Stage 4: VLM on graph pages for missing specs ──────────────
    if missing:
        print(f'\n[Stage 4] VLM graph extraction for: {missing}')

        # Step 4a: LLM finds which pages have the relevant graphs
        graph_page_info = find_graph_pages(raw_text, missing)
        print(f'  LLM identified graph locations: {graph_page_info}')

        for entry in graph_page_info:
            spec_name  = entry.get('spec')
            pages      = entry.get('pages', [])
            x_axis     = entry.get('x_axis', 'input voltage')
            typical_x  = entry.get('typical_x', 'typical operating point')

            if spec_name not in specs or specs[spec_name]['value']:
                continue  # already filled

            for pg in pages[:3]:
                print(f'  Scanning page {pg} for "{spec_name}" (x={typical_x})...')
                try:
                    b64    = render_page_to_base64(filepath, pg)
                    result = extract_spec_from_graph_vlm(
                        b64, spec_name, x_axis, typical_x, component_type
                    )
                    val = result.get('value')
                    is_null = (not val or str(val).strip().lower() == 'null')
                    if not is_null:
                        condition = result.get('condition', '')
                        specs[spec_name] = {
                            'value': val,
                            'source': f'graph_vlm_p{pg}',
                            'condition': condition
                        }
                        print(f'    ✅ {spec_name} = {val} @ {condition}')
                        break
                    else:
                        print(f'    ✗ Not readable on page {pg}')
                except Exception as e:
                    print(f'    ⚠ VLM error page {pg}: {e}')

        missing = get_missing(specs)
    else:
        print('\n[Stage 4] Skipped — no missing specs after text extraction')

    # ── Stage 5: Expand to next pages if still missing ─────────────
    if missing and total_pages > CHUNK_SIZE:
        start = CHUNK_SIZE
        while missing and start < total_pages:
            end = start + CHUNK_SIZE
            print(f'\n[Stage 5] Expanding to pages {start+1}–{min(end, total_pages)}')
            r2, t2, _ = extract_pdf_chunk(filepath, start, end)
            tt2 = tables_to_text(t2)

            # Try text extraction on new chunk for missing specs only
            new_specs, _, _ = extract_fixed_specs(tt2, r2, component_type, missing)
            for name, meta in new_specs.items():
                if meta.get('value') and name in specs and not specs[name]['value']:
                    specs[name] = meta
                    print(f'  ✅ {name} = {meta["value"]} (pages {start+1}–{min(end,total_pages)})')

            # Also run VLM on figure pages in this chunk
            missing = get_missing(specs)
            if missing:
                fig_pages_ext = get_figure_pages(filepath, start, end)
                for pg in fig_pages_ext[:4]:
                    if not missing:
                        break
                    try:
                        b64 = render_page_to_base64(filepath, pg)
                        vlm_result = extract_specs_vlm(b64, missing, component_type)
                        for spec_name, val in vlm_result.items():
                            if val and spec_name in specs and not specs[spec_name]['value']:
                                specs[spec_name] = {'value': val, 'source': f'graph_vlm_p{pg}'}
                                print(f'  ✅ {spec_name} = {val} (VLM page {pg})')
                        missing = get_missing(specs)
                    except Exception as e:
                        print(f'  ⚠ VLM error page {pg}: {e}')

            start = end

        # Accumulate all text+tables seen for dynamic extraction later
        full_raw   = raw_text + r2
        full_table = table_text + tt2
    else:
        full_raw   = raw_text
        full_table = table_text

    # ── Stage 6: Dynamic extraction for still-missing + extra specs ─
    print('\n[Stage 6] Dynamic spec extraction')
    extra_specs = extract_dynamic_specs(
        full_table, full_raw, component_type,
        list(specs.keys())
    )
    print(f'  Found {len(extra_specs)} additional specs: {[s["name"] for s in extra_specs]}')

    # If a missing fixed spec was found dynamically, promote it
    for item in extra_specs:
        if item['name'] in specs and not specs[item['name']]['value']:
            specs[item['name']] = {'value': item['value'], 'source': 'dynamic_llm'}

    # Build extra_specs dict (only truly new fields)
    extra = {
        item['name']: {'value': item['value'], 'source': 'dynamic_llm'}
        for item in extra_specs
        if item['name'] not in specs
    }

    # ── Assemble final record ──────────────────────────────────────
    record = {
        'pdf_hash':       sha,
        'filename':       os.path.basename(filepath),
        'part_name':      part_name or 'Unknown',
        'manufacturer':   manufacturer or 'Unknown',
        'component_type': component_type,
        'total_pages':    total_pages,
        'extracted_at': datetime.utcnow().isoformat(),
        'specs':          specs,
        'extra_specs':    extra,
    }

    # Print extraction summary
    found   = [k for k, v in specs.items() if v.get('value')]
    still_missing = [k for k, v in specs.items() if not v.get('value')]
    print(f'\n── Extraction Summary ──────────────────────────────')
    print(f'  Part:      {record["part_name"]} ({record["manufacturer"]})')
    print(f'  Type:      {component_type}')
    print(f'  Found:     {len(found)}/{len(specs)} fixed specs')
    print(f'  Missing:   {still_missing if still_missing else "None"}')
    print(f'  Extra:     {list(extra.keys())}')

    # Source breakdown (useful for project review)
    from collections import Counter
    sources = Counter(v['source'] for v in specs.values() if v.get('source'))
    sources.update(Counter(v['source'] for v in extra.values() if v.get('source')))
    print(f'  Sources:   {dict(sources)}')

    return record


print('✅ Pipeline defined')

✅ Pipeline defined


In [ ]:
# ── CELL 9: Upload PDF & run ───────────────────────────────────────────────────
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print('Uploaded:', pdf_filename)

Saving testSheet.pdf to testSheet.pdf
Uploaded: testSheet.pdf


In [ ]:
# ── CELL 10: Execute pipeline ──────────────────────────────────────────────────
result = run_pipeline(pdf_filename)

# ── Save to MongoDB ────────────────────────────────────────────────────────────
save_to_db(result)

# ── Save to JSON file ──────────────────────────────────────────────────────────
out_filename = f"{result['part_name'].replace(' ', '_')}_specs.json"
with open(out_filename, 'w') as f:
    json.dump(result, f, indent=2)
print(f'\n✅ JSON saved: {out_filename}')

# Download the JSON in Colab
files.download(out_filename)

# Pretty print final output
print('\n── Final JSON ──────────────────────────────────────────')
print(json.dumps(result, indent=2))


Processing: testSheet.pdf

[Stage 0] Checking cache (hash: 03631e58b412...)
  Cache miss — running full pipeline

[Stage 1] Extracting pages 1–25
  Pages: 17/17 | Tables: 31

[Stage 2] Detecting component type
  → LDO Regulator

[Stage 3] Fixed spec extraction (text + tables)
  Extracted: 4/5 | Missing: ['Dropout Voltage']

[Stage 4] VLM graph extraction for: ['Dropout Voltage']
  LLM identified graph locations: [{'spec': 'Dropout Voltage', 'pages': [5, 6], 'x_axis': 'Input Voltage', 'typical_x': '100mA'}]
  Scanning page 5 for "Dropout Voltage" (x=100mA)...
    ✅ Dropout Voltage = 160mV @ TJ = 125°C

[Stage 6] Dynamic spec extraction
  Found 5 additional specs: ['Operating Junction Temperature Range', 'Package Type', 'Number of Pins', 'Quiescent Current in Standby Mode', 'Output Noise Voltage']

── Extraction Summary ──────────────────────────────
  Part:      TPS76201 (Texas Instruments)
  Type:      LDO Regulator
  Found:     5/5 fixed specs
  Missing:   None
  Extra:     ['Operati

/tmp/ipykernel_7250/2422080200.py:164: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'extracted_at': datetime.utcnow().isoformat(),


✅ Saved to MongoDB: TPS76201 (LDO Regulator)

✅ JSON saved: TPS76201_specs.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


── Final JSON ──────────────────────────────────────────
{
  "pdf_hash": "03631e58b412bfd0a181b235a02428f8d81d3a9bab48206e89ac916c43935c7f",
  "filename": "testSheet.pdf",
  "part_name": "TPS76201",
  "manufacturer": "Texas Instruments",
  "component_type": "LDO Regulator",
  "total_pages": 17,
  "extracted_at": "2026-04-14T09:53:58.754970",
  "specs": {
    "Input Voltage (Vin)": {
      "value": "2.7 V to 10 V",
      "source": "text_llm"
    },
    "Output Voltage (Vout)": {
      "value": "0.7 V to 5.5 V",
      "source": "text_llm"
    },
    "Output Current (Iout)": {
      "value": "100 mA",
      "source": "text_llm"
    },
    "Dropout Voltage": {
      "value": "160mV",
      "source": "graph_vlm_p5",
      "condition": "TJ = 125\u00b0C"
    },
    "Quiescent Current (Iq)": {
      "value": "23 \u00b5A",
      "source": "text_llm"
    }
  },
  "extra_specs": {
    "Operating Junction Temperature Range": {
      "value": "-40\u00b0C to 125\u00b0C",
      "source": "dynamic_ll

In [ ]:
# ── CELL 11: Query MongoDB (optional) ─────────────────────────────────────────
# Fetch all stored components
all_docs = list(col.find({}, {'_id': 0, 'part_name': 1, 'component_type': 1,
                               'manufacturer': 1, 'extracted_at': 1}))
print(f'Total components in DB: {len(all_docs)}')
for d in all_docs:
    print(f"  {d['part_name']:20s} | {d['component_type']:20s} | {d['manufacturer']}")

Total components in DB: 5
  DA7212               | Audio Codec          | Renesas Electronics
  TC1263               | LDO Regulator        | Microchip Technology Inc.
  TPS76201             | LDO Regulator        | Texas Instruments
  LQW15AN62NG00        | Inductor             | Murata Manufacturing Co., Ltd.
  ADXL345              | Accelerometer        | Analog Devices
